# VGenC Top Teams Scraper
Scrapes team data from pokepaste links listed on vgenc.net/top-teams.

In [10]:
import requests
from bs4 import BeautifulSoup
import pickle
import time
import re
import os

In [11]:
# ========================== TOGGLES ==========================
# Set to True to only scrape teams with full pokepastes (ppf=1, i.e. EVs available)
# Set to False to scrape ALL teams with a pokepaste link
ONLY_FULL_PASTE = False

# Set to True to include EV entries in each pokemon vector
# Set to False to omit the 6 EV fields from the output
INCLUDE_EVS = False
# =============================================================

In [12]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
OUTPUT_PATH = os.path.join(NOTEBOOK_DIR, "vgenc_team_vectors.pkl")

# Fetch the team index from the static JSON (no scraping needed for the list)
print("Fetching team index from vgenc.net...")
resp = requests.get("https://vgenc.net/static/top-teams-data.json", timeout=15)
resp.raise_for_status()
all_entries = resp.json()
print(f"Total teams in index: {len(all_entries)}")

# Filter based on toggle
if ONLY_FULL_PASTE:
    entries = [t for t in all_entries if t.get("ppf") == 1 and t.get("pp")]
else:
    entries = [t for t in all_entries if t.get("pp")]

paste_urls = [t["pp"].strip() for t in entries]
print(f"Pokepaste URLs to scrape: {len(paste_urls)}")

Fetching team index from vgenc.net...
Total teams in index: 2613
Pokepaste URLs to scrape: 2613


In [13]:
EV_KEYS = ['HP', 'Atk', 'Def', 'SpA', 'SpD', 'Spe']

def parse_pokemon_block(text):
    """Parse a single pokemon's text block into the desired vector."""
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return None

    # Line 1: Name @ Item
    first_line = lines[0]
    if ' @ ' in first_line:
        name, item = first_line.split(' @ ', 1)
    else:
        name = first_line
        item = 'NONE'

    ability = 'NONE'
    evs = {k: 0 for k in EV_KEYS}
    moves = []

    for line in lines[1:]:
        if line.startswith('Ability:'):
            ability = line.split('Ability:', 1)[1].strip()
        elif line.startswith('EVs:'):
            ev_str = line.split('EVs:', 1)[1].strip()
            for part in ev_str.split('/'):
                part = part.strip()
                match = re.match(r'(\d+)\s+(\w+)', part)
                if match:
                    val, stat = int(match.group(1)), match.group(2)
                    if stat in evs:
                        evs[stat] = val
        elif line.startswith('- '):
            moves.append(line[2:].strip())

    # Pad moves to 4 with NONE
    while len(moves) < 4:
        moves.append('NONE')
    moves = moves[:4]

    if INCLUDE_EVS:
        return [
            name.strip(), ability,
            evs['HP'], evs['Atk'], evs['Def'],
            evs['SpA'], evs['SpD'], evs['Spe'],
            item.strip(),
            moves[0], moves[1], moves[2], moves[3]
        ]
    else:
        return [
            name.strip(), ability,
            item.strip(),
            moves[0], moves[1], moves[2], moves[3]
        ]

In [14]:
def scrape_pokepaste(url):
    """Scrape a pokepaste URL and return a list of pokemon vectors."""
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    team = []
    for pre in soup.find_all('pre'):
        text = pre.get_text()
        if text.strip():
            parsed = parse_pokemon_block(text)
            if parsed:
                team.append(parsed)
    return team

In [15]:
all_teams = []
failed_urls = []

for i, url in enumerate(paste_urls):
    if (i + 1) % 50 == 0 or i == 0:
        print(f"Scraping {i + 1}/{len(paste_urls)}: {url}")
    try:
        team = scrape_pokepaste(url)
        if team:
            all_teams.append(team)
        else:
            failed_urls.append((url, 'empty parse'))
    except Exception as e:
        failed_urls.append((url, str(e)))
    time.sleep(0.3)  # be polite to the server

print(f"\nDone! Scraped {len(all_teams)} teams successfully.")
if failed_urls:
    print(f"Failed on {len(failed_urls)} URLs:")
    for url, reason in failed_urls[:10]:
        print(f"  {url} — {reason}")

Scraping 1/2613: https://pokepast.es/54e62febd2e94b0e
Scraping 50/2613: https://pokepast.es/7984493c1c82e685
Scraping 100/2613: https://pokepast.es/38ece1daaef4aa8d
Scraping 150/2613: https://pokepast.es/3f37ead9585e8a0d
Scraping 200/2613: https://pokepast.es/ac667cbbafae1071
Scraping 250/2613: https://pokepast.es/fbef06931d2e689b
Scraping 300/2613: https://pokepast.es/d13b55b87362e27f
Scraping 350/2613: https://pokepast.es/b03a6163ad3427cd
Scraping 400/2613: https://pokepast.es/0ca229adff0b60e0
Scraping 450/2613: https://pokepast.es/c21d7806a7b7a6a5
Scraping 500/2613: https://pokepast.es/8520daccf6b38180
Scraping 550/2613: https://pokepast.es/a14b9be6620c816a
Scraping 600/2613: https://pokepast.es/df56667e58cdadda
Scraping 650/2613: https://pokepast.es/5da82b941b204302
Scraping 700/2613: https://pokepast.es/9076a39d56e32d9a
Scraping 750/2613: https://pokepast.es/3183df80979de14f
Scraping 800/2613: https://pokepast.es/b1b395c5fd4a43a2
Scraping 850/2613: https://pokepast.es/022ae5b372f2

In [20]:
len(all_teams[0])

6

In [22]:
for t in all_teams:
    if len(t) != 6:
        print(t)

[['Blastoise-Mega', 'Rain Dish', 'Blastoisinite', 'Shell Smash', 'Surf', 'Dark Pulse', 'Ice Beam'], ['Blastoise-Mega', 'Rain Dish', 'Blastoisinite', 'Shell Smash', 'Surf', 'Dark Pulse', 'Substitute'], ['Lucario-Mega', 'Inner Focus', 'Lucarionite', 'Close Combat', 'Bullet Punch', 'Ice Punch', 'Earthquake'], ['Lucario-Mega', 'Inner Focus', 'Lucarionite', 'Close Combat', 'Meteor Mash', 'Extreme Speed', 'Earthquake'], ['Lucario-Mega', 'Inner Focus', 'Lucarionite', 'Swords Dance', 'Close Combat', 'Bullet Punch', 'Earthquake'], ['Lucario-Mega', 'Inner Focus', 'Lucarionite', 'Nasty Plot', 'Vacuum Wave', 'Flash Cannon', 'Dark Pulse'], ['Starmie-Mega', 'Natural Cure', 'Starminite', 'Flip Turn', 'Aqua Jet', 'Psycho Cut', 'Ice Beam'], ['Starmie-Mega', 'Natural Cure', 'Starminite', 'Bulk Up', 'Liquidation', 'Psycho Cut', 'Aqua Jet'], ['Meganium-Mega', 'Leaf Guard', 'Meganiumite', 'Solar Beam', 'Weather Ball', 'Dazzling Gleam', 'Synthesis'], ['Scizor-Mega', 'Technician', 'Scizorite', 'Swords Dance'

In [24]:
full_teams = all_teams.copy()

In [25]:
all_teams = [t for t in full_teams if len(t) == 6]

In [26]:
len(all_teams)

2611

In [27]:
# Preview first team
if all_teams:
    print(f"Example team (first scraped):")
    for mon in all_teams[0]:
        print(f"  {mon}")

Example team (first scraped):
  ['Garchomp', 'Rough Skin', 'Haban Berry', 'Dragon Claw', 'Earthquake', 'Rock Slide', 'Protect']
  ['Aerodactyl', 'Unnerve', 'Aerodactylite', 'Rock Slide', 'Dual Wingbeat', 'Tailwind', 'Protect']
  ['Venusaur', 'Chlorophyll', 'Venusaurite', 'Energy Ball', 'Sludge Bomb', 'Sleep Powder', 'Protect']
  ['Torkoal', 'Drought', 'Sitrus Berry', 'Eruption', 'Heat Wave', 'Earth Power', 'Protect']
  ['Kingambit', 'Defiant', 'Chople Berry', 'Kowtow Cleave', 'Sucker Punch', 'Iron Head', 'Low Kick']
  ['Basculegion-F (F)', 'Adaptability', 'Choice Scarf', 'Shadow Ball', 'Muddy Water', 'Hydro Pump', 'Ice Beam']


In [28]:
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(all_teams, f)

print(f"Saved {len(all_teams)} teams to {OUTPUT_PATH}")
print(f"Structure: list of {len(all_teams)} teams, each team is a list of pokemon vectors")
if all_teams:
    print(f"Each pokemon vector has {len(all_teams[0][0])} entries")

Saved 2611 teams to /Users/xanderdeanhardt/Documents/Claude Code Folder/Data_science_files/MS ADS/ML 2/Final Project/vgenc_team_vectors.pkl
Structure: list of 2611 teams, each team is a list of pokemon vectors
Each pokemon vector has 7 entries


In [29]:
# Verify reload
with open(OUTPUT_PATH, 'rb') as f:
    loaded = pickle.load(f)
print(f"Reloaded {len(loaded)} teams from {OUTPUT_PATH}")
print(f"First team, first pokemon: {loaded[0][0]}")

Reloaded 2611 teams from /Users/xanderdeanhardt/Documents/Claude Code Folder/Data_science_files/MS ADS/ML 2/Final Project/vgenc_team_vectors.pkl
First team, first pokemon: ['Garchomp', 'Rough Skin', 'Haban Berry', 'Dragon Claw', 'Earthquake', 'Rock Slide', 'Protect']
